# Thesis Results Analysis

This notebook combines two tasks (A and B) into one reproducible workflow for thesis: implementation, verification, parallel execution, visualization, and export of results.

In [ ]:
# 1) Import libraries
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor, as_completed

print('Libraries imported')

In [ ]:
# 2) Configuration & parameters
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

OUT_ROOT = os.path.abspath('results/gt_experiment')
ensure_dirs = lambda p: os.makedirs(p, exist_ok=True)
MODE_PARALLEL = True
print('Config set')

In [ ]:
# 3) Prepare data for Task A (Improved analyzer results)
improved_path = os.path.join(OUT_ROOT, 'stance_results_improved.csv')
if os.path.exists(improved_path):
    df_a = pd.read_csv(improved_path)
    print('Loaded Task A data:', improved_path, 'rows=', len(df_a))
else:
    print('Task A results not found at', improved_path)
    df_a = pd.DataFrame({'post_id':[], 'full_text_comments':[], 'stance':[], 'stance_confidence':[]})

# Basic cleaning
if not df_a.empty:
    df_a['text_norm'] = df_a['full_text_comments'].astype(str).str.lower().str.strip()


In [ ]:
# 4) Implementation Task A: helper functions
from typing import Dict, Tuple

def analyze_task_a(df: pd.DataFrame) -> Dict[str, float]:
    """Compute simple metrics for Task A results."""
    res = {'total': len(df)}
    if len(df) == 0:
        res.update({'Positive':0,'Negative':0,'Neutral':0})
        return res
    counts = df['stance'].value_counts().to_dict()
    for k in ['Positive','Negative','Neutral']:
        res[k] = counts.get(k,0)
    return res

# Docstring test
print('Task A helper defined')

In [ ]:
# 5) Run & verify Task A
metrics_a = analyze_task_a(df_a)
print('Task A metrics:', metrics_a)
# Simple assertion: total equals sum of class counts
if metrics_a['total'] > 0:
    assert metrics_a['total'] == (metrics_a['Positive'] + metrics_a['Negative'] + metrics_a['Neutral']), 'Count mismatch in Task A'
print('Task A verification passed')

In [ ]:
# 6) Prepare data for Task B (simulate transformer or alternative)
# We simulate Task B by reusing improved results as "alternative" outputs or creating synthetic labels
alt_path = os.path.join('results','full_experiment','stance_results_improved.csv')
if os.path.exists(alt_path):
    df_b = pd.read_csv(alt_path)
    df_b['text_norm'] = df_b['full_text_comments'].astype(str).str.lower().str.strip()
    print('Loaded Task B data (simulated):', alt_path, 'rows=', len(df_b))
else:
    # create synthetic small dataset
    df_b = df_a.copy()
    if not df_b.empty:
        df_b['stance'] = df_b['stance'].apply(lambda s: s if np.random.rand() > 0.2 else np.random.choice(['Positive','Negative','Neutral']))
    print('Created simulated Task B dataset')


In [ ]:
# 7) Implementation Task B: helper functions

def analyze_task_b(df: pd.DataFrame) -> Dict[str, float]:
    """Compute metrics for Task B (simulated transformer outputs)."""
    return analyze_task_a(df)

print('Task B helper defined')

In [ ]:
# 8) Run & verify Task B
metrics_b = analyze_task_b(df_b)
print('Task B metrics:', metrics_b)
if metrics_b['total'] > 0:
    assert metrics_b['total'] == (metrics_b['Positive'] + metrics_b['Negative'] + metrics_b['Neutral']), 'Count mismatch in Task B'
print('Task B verification passed')

In [ ]:
# 9) Combine execution: sequential and parallel

def run_sequential():
    a = analyze_task_a(df_a)
    b = analyze_task_b(df_b)
    return {'A': a, 'B': b}


def run_parallel():
    results = {}
    with ThreadPoolExecutor(max_workers=2) as ex:
        futures = {
            ex.submit(analyze_task_a, df_a): 'A',
            ex.submit(analyze_task_b, df_b): 'B'
        }
        for fut in as_completed(futures):
            key = futures[fut]
            try:
                results[key] = fut.result()
            except Exception as e:
                results[key] = {'error': str(e)}
    return results

print('Sequential run:')
seq = run_sequential()
print(seq)
print('\nParallel run:')
par = run_parallel()
print(par)


In [ ]:
# 10) Visualization and comparison
import matplotlib.pyplot as plt

# Metrics comparison bar chart
labels = ['Positive','Negative','Neutral']
vals_a = [metrics_a.get(k,0) for k in labels]
vals_b = [metrics_b.get(k,0) for k in labels]

x = np.arange(len(labels))
width = 0.35
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(x - width/2, vals_a, width, label='Improved')
ax.bar(x + width/2, vals_b, width, label='Alternative')
ax.set_ylabel('Counts')
ax.set_title('Per-class counts: Improved vs Alternative')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
plt.tight_layout()
plot_path = os.path.join(OUT_ROOT, 'figs','comparison_counts.png')
ensure_dirs(os.path.dirname(plot_path))
fig.savefig(plot_path)
plt.show()

print('Saved comparison plot to', plot_path)


In [ ]:
# 11) Unit tests and assertions (quick checks)
# Quick assertions for key functions
if len(df_a) > 0:
    assert 'stance' in df_a.columns
    assert isinstance(metrics_a['total'], int)
if len(df_b) > 0:
    assert 'stance' in df_b.columns
print('Quick unit checks passed')

In [ ]:
# 12) Save / export results
out_zip = os.path.abspath('results/gt_experiment/thesis_attachment_sample.zip')
import zipfile
with zipfile.ZipFile(out_zip, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    # include key artifacts
    for p in ['stance_results_improved.csv','detailed_metrics.csv','confusion_matrix.csv','error_examples.csv','REPORT.md']:
        fp = os.path.join(OUT_ROOT, p)
        if os.path.exists(fp):
            z.write(fp, arcname=os.path.join('gt_experiment', p))
    # include figs
    figs_dir = os.path.join(OUT_ROOT, 'figs')
    if os.path.exists(figs_dir):
        for f in os.listdir(figs_dir):
            z.write(os.path.join(figs_dir, f), arcname=os.path.join('gt_experiment','figs', f))

print('Created attachment zip:', out_zip)
